# Bayesian Networks and Autoregressive Language Models

Everything for this lab in one notebook: the write up, the code, the runs and the tests. Run all cells from top to bottom.

## Q1: Why the chain rule decomposition helps generation

It turns one impossible to estimate joint distribution into a product of "next word" distributions, each conditioned only on what has already been produced. Generation then becomes a loop: sample X1, then sample X2 given X1, and so on. Each step is a small, ordinary prediction problem, and the order matches how text is actually written, left to right.

## Q2: The independence assumption of the chain network

X1 → X2 → X3 → X4 assumes each word is conditionally independent of all earlier words given the immediately previous word:

P(X_t | X_1, ..., X_{t-1}) = P(X_t | X_{t-1}),  equivalently  X_t ⊥ X_1, ..., X_{t-2} | X_{t-1}.

## Q3: Conditional probability tables

Counts from the six training sentences (`the` appears 12 times as a context: followed by cat 3, dog 3, mat 2, rug 2, park 2):

| Current word | Next word distribution |
|---|---|
| `<START>` | the 1.000 |
| the | cat 0.250, dog 0.250, mat 0.167, rug 0.167, park 0.167 |
| cat | sat 0.667, ran 0.333 |
| dog | sat 0.667, ran 0.333 |
| sat | on 1.000 |
| ran | to 1.000 |
| on | the 1.000 |
| to | the 1.000 |
| mat, rug, park | `<END>` 1.000 |

**Zero probability transitions** (examples): P(the given the), P(`<END>` given the), P(mat given cat), P(on given ran), P(to given sat), P(park given on). In fact P(park given on) = 0 and P(mat given to) = 0 in the true sentences, but the bigram model cannot tell "the" after "on" from "the" after "to", so it will happily generate "the cat ran to the mat" anyway. That is the cost of the Markov assumption. Of the 11 × 12 = 132 possible (context, next) pairs for observed contexts, only 17 are nonzero.

## Part V: Prompt Used

> Write a simple Python implementation of a first-order autoregressive language model. The model should: 1. take a list of tokenised sentences as training data; 2. count transitions between consecutive tokens; 3. construct the conditional distribution P(X_t given X_{t-1}); 4. display the probabilities for a specified previous token; 5. predict the most probable next token; 6. generate a sentence by repeatedly sampling the next token; 7. stop when the `<END>` token is generated. Do not use a machine-learning library or a pretrained language model. Use ordinary Python data structures and random sampling. Support a greedy mode and a sampling mode, cap generation length, and handle words never seen as a context.

### Training data and tokeniser

In [1]:
"""Training data shared by both models."""

START, END = "<START>", "<END>"

SENTENCES = [
    "the cat sat on the mat",
    "the cat sat on the rug",
    "the dog sat on the mat",
    "the dog ran to the park",
    "the cat ran to the park",
    "the dog sat on the rug",
]


def tokenise(sentences, order=1):
    """Lower case, split on spaces, pad with `order` START tokens and one END."""
    return [[START] * order + s.lower().split() + [END] for s in sentences]

### First order (bigram) model

In [2]:
"""
Lab 4 Part V: first-order autoregressive language model (bigram model).

Bayesian network:  X1 -> X2 -> X3 -> ... -> XT
Model:             P(X_t | X_{t-1}) estimated by counting
                   P(w_j | w_i) = C(w_i, w_j) / sum_k C(w_i, w_k)

No ML library is used. Only dicts, Counter and random.
"""

import random
from collections import Counter, defaultdict


class BigramLM:
    def __init__(self, sentences):
        # transition counts C(prev, next) live here  (Question 4)
        self.counts = defaultdict(Counter)
        for tokens in sentences:
            for prev, nxt in zip(tokens, tokens[1:]):
                self.counts[prev][nxt] += 1
        # conditional distribution P(X_t | X_{t-1}) is computed here  (Question 5)
        self.probs = {
            prev: {w: c / sum(ctr.values()) for w, c in ctr.items()}
            for prev, ctr in self.counts.items()
        }

    def distribution(self, prev):
        """P(next | prev). Empty dict for a word never seen as a context (Question 7)."""
        return self.probs.get(prev, {})

    def most_probable(self, prev):
        dist = self.distribution(prev)
        return max(dist, key=dist.get) if dist else None

    def sample(self, prev, rng=random):
        """Draw the next word in proportion to its probability (Question 6)."""
        dist = self.distribution(prev)
        if not dist:
            return END  # unseen context: stop instead of crashing
        words = list(dist)
        return rng.choices(words, weights=[dist[w] for w in words])[0]

    def generate(self, mode="sample", max_len=20, rng=random):
        """mode='sample' draws from the distribution, mode='greedy' takes the argmax."""
        prev, out = START, []
        for _ in range(max_len):
            nxt = self.most_probable(prev) if mode == "greedy" else self.sample(prev, rng)
            if nxt == END:
                break
            out.append(nxt)
            prev = nxt
        return " ".join(out)

    def sentence_probability(self, sentence):
        tokens = [START] + sentence.split() + [END]
        p = 1.0
        for prev, nxt in zip(tokens, tokens[1:]):
            p *= self.distribution(prev).get(nxt, 0.0)
        return p


def print_cpt(model, words):
    for w in words:
        dist = model.distribution(w)
        row = ", ".join(f"P({v}|{w})={p:.3f}" for v, p in sorted(dist.items(), key=lambda x: -x[1]))
        print(f"{w:>8}: {row}")

In [3]:
rng = random.Random(42)
lm = BigramLM(tokenise(SENTENCES, order=1))

print("Conditional probability tables  P(next | current)")
print_cpt(lm, [START, "the", "cat", "dog", "sat", "ran", "on", "to"])
vocab = sorted({w for ctr in lm.counts.values() for w in ctr} | set(lm.counts))
print(f"\nVocabulary ({len(vocab)}): {vocab}")

print("\nMost probable next word")
for w in ["the", "cat", "dog", "sat", "ran", "on", "to"]:
    print(f"  argmax P(w | {w}) = {lm.most_probable(w)}")

print("\nGreedy generation (5)")
for _ in range(5):
    print("  ", lm.generate("greedy"))
print("\nSampling generation (20)")
for _ in range(20):
    print("  ", lm.generate("sample", rng=rng))

print("\nUnseen context 'bird':", lm.distribution("bird"), "-> sample returns", lm.sample("bird"))

Conditional probability tables  P(next | current)
 <START>: P(the|<START>)=1.000
     the: P(cat|the)=0.250, P(dog|the)=0.250, P(mat|the)=0.167, P(rug|the)=0.167, P(park|the)=0.167
     cat: P(sat|cat)=0.667, P(ran|cat)=0.333
     dog: P(sat|dog)=0.667, P(ran|dog)=0.333
     sat: P(on|sat)=1.000
     ran: P(to|ran)=1.000
      on: P(the|on)=1.000
      to: P(the|to)=1.000

Vocabulary (12): ['<END>', '<START>', 'cat', 'dog', 'mat', 'on', 'park', 'ran', 'rug', 'sat', 'the', 'to']

Most probable next word
  argmax P(w | the) = cat
  argmax P(w | cat) = sat
  argmax P(w | dog) = sat
  argmax P(w | sat) = on
  argmax P(w | ran) = to
  argmax P(w | on) = the
  argmax P(w | to) = the

Greedy generation (5)
   the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
   the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
   the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
   the cat sat on the cat sat on the cat sat on the cat s

## Part VI: Inspecting the Code

**Q4.** In `BigramLM.__init__`, `self.counts`: a `defaultdict(Counter)` where `self.counts[prev][next]` is C(prev, next).

**Q5.** In the dict comprehension that builds `self.probs` in `__init__`: each count is divided by the row total `sum(ctr.values())`.

**Q6.** `generate(mode="sample")` calls `sample()`, which uses `random.choices` with the probabilities as weights, so it **samples** from the distribution. `mode="greedy"` calls `most_probable()`, which always takes the argmax. Sampling picks "mat" after "the" about 16.7% of the time; greedy never does.

**Q7.** `distribution()` returns an empty dict for an unseen context. `sample()` then returns `<END>`, so generation stops instead of raising a `KeyError`. Example: `distribution("bird")` gives `{}` and `sample("bird")` gives `<END>`. This guard was asked for explicitly in the prompt, because the most common way to write `sample()` (indexing `self.probs[prev]` directly) raises a `KeyError` for an unseen word. (`<END>` is never a context during generation because generation stops when it appears.)

## Part VII: Normalisation Test

Every row sums to 1.000000 for both models (full printout in the comparison cell further down). The test cell checks this automatically, plus: known values such as P(cat given the) = 3/12, every probability in (0, 1], greedy output is deterministic, sampling frequency of "cat" after "the" over 20,000 draws is within 0.01 of 0.25, and generation always terminates.

**Q8.** A total of 0.87 means the row is not a probability distribution: about 13% of the mass is missing. Likely causes: dividing by the wrong total (for example the word's overall frequency, including occurrences at the end of a sentence that have no following token, or including `<END>` in the denominator but not the numerator), or a loop that skips the last transition in each sentence, so transitions into `<END>` are never counted. Either way, sampling would be biased and sentence probabilities would be wrong.

## Part VIII: Next Word Prediction

| Context | Most probable next word |
|---|---|
| the | cat (tied with dog at 0.25) |
| cat | sat |
| dog | sat |
| sat | on |
| ran | to |
| on | the |
| to | the |

**Q9.** Not always. After "the", a person might expect "mat" or a noun that fits the earlier words, but the model says "cat" purely because "the cat" and "the dog" are the most frequent bigrams, and picks "cat" over "dog" only because of the order the counts were inserted. A probability model reflects frequencies in its (tiny) training data and the context it is allowed to see. Human expectation uses meaning, grammar, world knowledge and the whole sentence so far.

## Parts IX and X: Generation, Greedy vs Sampling


First order, greedy (all 5 identical):
```
the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
```
First order, sampling (examples):
```
the rug
the dog sat on the mat
the cat sat on the park
the dog sat on the dog sat on the mat
the cat sat on the dog sat on the dog sat on the cat ran to the dog ran to
```

**Q10.** Sampling produces far more variation: 56 distinct sentences out of 200 samples, against exactly 1 for greedy. Greedy is deterministic, so the same context always gives the same next word. Here it also gets stuck in a loop: argmax after "the" is "cat", never a noun that leads to `<END>`, so it cycles "the cat sat on" until the 20 token cap stops it. Sampling explores lower probability branches, which gives variety but also nonsense like "the park" or "the cat sat on the park".

## Part XI: Q11, First vs Second Order

| | First order | Second order |
|---|---|---|
| Graph | Each X_t has one parent X_{t-1} | Each X_t has two parents X_{t-2}, X_{t-1} (a v structure into X_t) |
| CPT | Rows indexed by one word: up to V rows of V entries | Rows indexed by word pairs: up to V² rows of V entries |
| Context | One previous word | Two previous words, so it knows "on the" vs "to the" |
| Data needed | Must observe each word pair | Must observe each word triple, which is far rarer, so far more data is needed |

### Second order (trigram) model

In [4]:
"""
Lab 4 Part XII: second-order autoregressive language model (trigram model).

Bayesian network:  X_{t-2} -> X_t <- X_{t-1}
Model:             P(X_t | X_{t-2}, X_{t-1}) = C(a, b, w) / sum_k C(a, b, k)

Each sentence is padded with TWO <START> tokens so that X1 and X2 also have a
two-token context:  P(X1 | START, START),  P(X2 | START, X1).
"""

import random
from collections import Counter, defaultdict


class TrigramLM:
    def __init__(self, sentences):
        # counts of observed triples, keyed by the 2-token context
        self.counts = defaultdict(Counter)
        for tokens in sentences:
            for a, b, c in zip(tokens, tokens[1:], tokens[2:]):
                self.counts[(a, b)][c] += 1
        self.probs = {
            ctx: {w: n / sum(ctr.values()) for w, n in ctr.items()}
            for ctx, ctr in self.counts.items()
        }

    def distribution(self, ctx):
        return self.probs.get(tuple(ctx), {})

    def most_probable(self, ctx):
        dist = self.distribution(ctx)
        return max(dist, key=dist.get) if dist else None

    def sample(self, ctx, rng=random):
        dist = self.distribution(ctx)
        if not dist:
            return END
        words = list(dist)
        return rng.choices(words, weights=[dist[w] for w in words])[0]

    def generate(self, mode="sample", max_len=20, rng=random):
        ctx, out = (START, START), []
        for _ in range(max_len):
            nxt = self.most_probable(ctx) if mode == "greedy" else self.sample(ctx, rng)
            if nxt == END:
                break
            out.append(nxt)
            ctx = (ctx[1], nxt)
        return " ".join(out)

In [5]:
rng = random.Random(42)
lm = TrigramLM(tokenise(SENTENCES, order=2))

print("Selected CPT rows  P(next | prev2, prev1)")
for ctx in [(START, "the"), ("the", "cat"), ("the", "dog"), ("cat", "sat"),
            ("dog", "ran"), ("on", "the"), ("to", "the")]:
    dist = lm.distribution(ctx)
    row = ", ".join(f"{w}={p:.3f}" for w, p in sorted(dist.items(), key=lambda x: -x[1]))
    print(f"  {ctx}: {row}")

print("\nGreedy generation (5)")
for _ in range(5):
    print("  ", lm.generate("greedy"))
print("\nSampling generation (20)")
for _ in range(20):
    print("  ", lm.generate("sample", rng=rng))

Selected CPT rows  P(next | prev2, prev1)
  ('<START>', 'the'): cat=0.500, dog=0.500
  ('the', 'cat'): sat=0.667, ran=0.333
  ('the', 'dog'): sat=0.667, ran=0.333
  ('cat', 'sat'): on=1.000
  ('dog', 'ran'): to=1.000
  ('on', 'the'): mat=0.500, rug=0.500
  ('to', 'the'): park=1.000

Greedy generation (5)
   the cat sat on the mat
   the cat sat on the mat
   the cat sat on the mat
   the cat sat on the mat
   the cat sat on the mat

Sampling generation (20)
   the cat sat on the rug
   the cat sat on the mat
   the dog sat on the mat
   the cat sat on the mat
   the dog ran to the park
   the dog sat on the rug
   the cat sat on the mat
   the cat sat on the rug
   the dog sat on the rug
   the dog ran to the park
   the dog ran to the park
   the cat sat on the rug
   the cat ran to the park
   the dog sat on the rug
   the dog sat on the rug
   the cat sat on the rug
   the cat ran to the park
   the dog sat on the mat
   the dog ran to the park
   the cat sat on the rug


## Part XIII: Comparison

Vocabulary V = 12 tokens (including `<START>` and `<END>`):

| Measure | First order | Second order |
|---|---|---|
| Nonzero parameters | 17 | 19 |
| Full CPT size (contexts × next words) | 144 | 1728 |
| Contexts observed | 11 | 15 |
| Possible contexts | 12 | 144 |
| Zero probability (unobserved) contexts | 1 | 129 |
| Zero probability CPT entries | 127 | 1709 |
| Distinct sentences in 200 samples | 56 | 6 |
| Novel sentences (not in training data) | 50 | 0 |

Second order examples (sampling): `the dog ran to the park`, `the cat sat on the rug`, `the cat ran to the park`. Greedy: always `the cat sat on the mat`.

Qualitatively the second order model is perfectly coherent: it never says "ran to the mat" because ("to", "the") is always followed by "park". But with only six training sentences, every two word context has at most a couple of continuations, so it can only reproduce the training sentences (6 distinct, 0 novel). The first order model is creative but often ungrammatical or semantically odd. This is a direct view of the bias vs variance trade off.

**Q12.** More context removes ambiguity: P(next given "on the") is much sharper than P(next given "the"), so predictions improve. But the number of possible contexts grows as V^n. Here the full CPT grows from 144 to 1,728 entries while the data stays the same, so 129 of the 144 possible contexts are never seen and the estimates for the rest come from one or two examples each. The model memorises rather than generalises. With real vocabularies (V ≈ 50,000) a trigram table would have about 1.25 × 10^14 entries, impossible to fill by counting.

### Normalisation check and comparison table

In [6]:
"""
Part VII normalisation printout + Part XIII comparison.
Writes outputs/generated_sentences.txt and outputs/comparison.md
"""
import random




def stats(model, vocab_size, ctx_len, rng, n=200):
    params = sum(len(d) for d in model.probs.values())
    possible_ctx = vocab_size ** ctx_len
    seen_ctx = len(model.probs)
    zero_entries = possible_ctx * vocab_size - params
    gens = [model.generate("sample", rng=rng) for _ in range(n)]
    train = {s for s in SENTENCES}
    novel = [g for g in set(gens) if g not in train]
    return {
        "nonzero parameters": params,
        "full CPT size (contexts x next words)": possible_ctx * vocab_size,
        "contexts observed": seen_ctx,
        "possible contexts": possible_ctx,
        "zero-probability contexts (never observed)": possible_ctx - seen_ctx,
        "zero-probability CPT entries": zero_entries,
        f"distinct sentences in {n} samples": len(set(gens)),
        "novel sentences (not in training data)": len(novel),
    }, gens, novel


def run_comparison():
    bi = BigramLM(tokenise(SENTENCES, 1))
    tri = TrigramLM(tokenise(SENTENCES, 2))

    print("Normalisation check, sum_v P(v | context):")
    for word in bi.probs:
        print(f"  {word:>8}  {sum(bi.probs[word].values()):.6f}")
    for ctx in tri.probs:
        print(f"  {str(ctx):>22}  {sum(tri.probs[ctx].values()):.6f}")

    # vocabulary of possible context tokens: words plus <START> (END never starts a context)
    vocab = {w for t in tokenise(SENTENCES, 1) for w in t}
    V = len(vocab)
    s1, g1, n1 = stats(bi, V, 1, random.Random(7))
    s2, g2, n2 = stats(tri, V, 2, random.Random(7))

    lines = ["| Measure | First order | Second order |", "|---|---|---|"]
    for k in s1:
        lines.append(f"| {k} | {s1[k]} | {s2[k]} |")
    table = "\n".join(lines)
    print("\n" + table)
    print("\nNovel bigram sentences:", sorted(n1))
    print("Novel trigram sentences:", sorted(n2))



run_comparison()

Normalisation check, sum_v P(v | context):
   <START>  1.000000
       the  1.000000
       cat  1.000000
       sat  1.000000
        on  1.000000
       mat  1.000000
       rug  1.000000
       dog  1.000000
       ran  1.000000
        to  1.000000
      park  1.000000
  ('<START>', '<START>')  1.000000
      ('<START>', 'the')  1.000000
          ('the', 'cat')  1.000000
          ('cat', 'sat')  1.000000
           ('sat', 'on')  1.000000
           ('on', 'the')  1.000000
          ('the', 'mat')  1.000000
          ('the', 'rug')  1.000000
          ('the', 'dog')  1.000000
          ('dog', 'sat')  1.000000
          ('dog', 'ran')  1.000000
           ('ran', 'to')  1.000000
           ('to', 'the')  1.000000
         ('the', 'park')  1.000000
          ('cat', 'ran')  1.000000

| Measure | First order | Second order |
|---|---|---|
| nonzero parameters | 17 | 19 |
| full CPT size (contexts x next words) | 144 | 1728 |
| contexts observed | 11 | 15 |
| possible contexts | 12 

### Tests

In [7]:
"""
Part VII: probabilistic invariants.  Run: python -m unittest -v
For every context w:  sum_v P(v | w) = 1
"""
import random
import unittest


class TestInvariants(unittest.TestCase):
    def setUp(self):
        self.bi = BigramLM(tokenise(SENTENCES, 1))
        self.tri = TrigramLM(tokenise(SENTENCES, 2))

    def test_bigram_rows_sum_to_one(self):
        for ctx, dist in self.bi.probs.items():
            self.assertAlmostEqual(sum(dist.values()), 1.0, places=9, msg=ctx)

    def test_trigram_rows_sum_to_one(self):
        for ctx, dist in self.tri.probs.items():
            self.assertAlmostEqual(sum(dist.values()), 1.0, places=9, msg=ctx)

    def test_known_values(self):
        self.assertAlmostEqual(self.bi.distribution("the")["cat"], 3 / 12)
        self.assertAlmostEqual(self.bi.distribution("cat")["sat"], 2 / 3)
        self.assertEqual(self.bi.distribution(START), {"the": 1.0})

    def test_probabilities_in_range(self):
        for model in (self.bi, self.tri):
            for dist in model.probs.values():
                for p in dist.values():
                    self.assertTrue(0.0 < p <= 1.0)

    def test_greedy_is_deterministic(self):
        outs = {self.bi.generate("greedy") for _ in range(10)}
        self.assertEqual(len(outs), 1)

    def test_sampling_frequency_matches_probability(self):
        rng = random.Random(0)
        n = 20000
        hits = sum(self.bi.sample("the", rng) == "cat" for _ in range(n))
        self.assertAlmostEqual(hits / n, 3 / 12, delta=0.01)

    def test_generation_terminates(self):
        rng = random.Random(1)
        for _ in range(200):
            self.assertLessEqual(len(self.bi.generate("sample", rng=rng).split()), 20)

    def test_unseen_context(self):
        self.assertEqual(self.bi.distribution("bird"), {})
        self.assertEqual(self.bi.sample("bird"), END)


unittest.main(argv=["notebook"], exit=False, verbosity=2)

test_bigram_rows_sum_to_one (__main__.TestInvariants.test_bigram_rows_sum_to_one) ... 

ok


test_generation_terminates (__main__.TestInvariants.test_generation_terminates) ... 

ok


test_greedy_is_deterministic (__main__.TestInvariants.test_greedy_is_deterministic) ... 

...

test_trigram_rows_sum_to_one (__main__.TestInvariants.test_trigram_rows_sum_to_one) ... 

ok


test_unseen_context (__main__.TestInvariants.test_unseen_context) ... 

ok


----------------------------------------------------------------------
Ran 8 tests in 0.044s

OK


## Part XV: Q13, Why Approach B

Approach B ("implement P(X_t given X_{t-1}) from transition counts with sampling") fixes the model before the code exists. That means:
* **Specified behaviour:** the LLM cannot quietly swap in a pretrained model, a neural network or a different smoothing scheme.
* **Understood representation:** I know the counts live in a nested dict and the CPT is the normalised version, so I can point to them in the code (Q4, Q5).
* **Validation:** I can compute expected values by hand (P(cat given the) = 3/12) and compare.
* **Invariants:** rows must sum to 1 and sample frequencies must match probabilities. These tests only exist because the model was stated mathematically.
* **Model vs implementation:** a bug (for example missing `<END>` counts) is an implementation error, not a property of the bigram model. Without a stated model I could not tell the two apart.

Approach A gives code whose behaviour I would have to reverse engineer before I could even decide whether it is correct.

## Q14: What the Bayesian network view added

* **Factorisation of the joint:** the chain of edges is exactly the product P(X1) Π P(X_t given X_{t-1}), which is what the code computes in `sentence_probability`.
* **Independence assumptions made visible:** the missing edge from X_{t-2} to X_t is the reason "the cat ran to the mat" is generated. Adding that edge (second order) fixes it.
* **Principled generation:** ancestral sampling in topological order (sample parents before children) is just left to right generation.
* **Effect of context:** adding a parent multiplies the CPT size by V, which explains both the improved coherence and the data sparsity measured above.
* **Testable specification:** every CPT row must be a distribution, which became the normalisation test.

## Reflection on LLM Use

The LLM wrote the counting, normalisation and sampling code correctly from the specification. The piece of code I inspected most closely was `sample()`, comparing it with the naive pattern that would fail on unseen contexts:

```python
# naive pattern: crashes with KeyError for an unseen word
def sample(self, prev):
    dist = self.probs[prev]
    return random.choices(list(dist), weights=list(dist.values()))[0]

# version kept in the bigram model
def sample(self, prev, rng=random):
    dist = self.distribution(prev)    # returns {} for unseen contexts
    if not dist:
        return END
    words = list(dist)
    return rng.choices(words, weights=[dist[w] for w in words])[0]
```

The `rng` argument makes runs reproducible with a seed, and the `max_len` cap turned out to be essential: without it greedy generation never terminates, because it loops on "the cat sat on". The unit test `test_generation_terminates` guards this. The tie between "cat" and "dog" after "the" was not mentioned by the LLM; I found it by reading the CPT.